In [1]:
import csv
import json
import os

def get_last_step(metrics_csv):
    with open(metrics_csv, newline='') as inf:
        rows = [r for r in csv.reader(inf)]
    assert rows[0] == ["epoch", "grad_norm", "lr-AdamW", "step", "train_loss_epoch", "train_loss_step", "val_loss_epoch", "val_loss_step"]
    final_step = rows[-1][3]
    return final_step

def read_scores(scores_json):
    if not os.path.exists(scores_json):
        return None
    with open(scores_json) as inf:
        data = json.load(inf)
    return data["BEST_VAL_BLEU_CHECKPOINT"]

In [2]:
import os
from tqdm import tqdm
experiments_dir = "/home/hatch5o6/nobackup/archive/CognateMT/PredictCognates/archive/CoNLL"
records = {}
for lang_pair in tqdm(os.listdir(experiments_dir)):
    if lang_pair == "archive" or "chosen" in lang_pair: continue

    assert lang_pair not in records
    records[lang_pair] = []

    lang_pair_path = os.path.join(experiments_dir, lang_pair)
    for exp_dir in os.listdir(lang_pair_path):
        if exp_dir == "mt_eval": continue
        exp_dir_path = os.path.join(lang_pair_path, exp_dir)
        if os.path.isdir(exp_dir_path):
            record = {
                "lang_pair": lang_pair,
                "model": exp_dir,
                "val_step_loss": os.path.join(exp_dir_path, "logs/lightning_logs/version_0/val_step_loss.png"),
                "train_step_loss": os.path.join(exp_dir_path, "logs/lightning_logs/version_0/train_step_loss.png"),
                "epoch_loss": os.path.join(exp_dir_path, "logs/lightning_logs/version_0/epoch_loss.png"),
                "hyperparams": os.path.join(exp_dir_path, "logs/lightning_logs/version_0/hparams.yaml"),
                "metrics": os.path.join(exp_dir_path, "logs/lightning_logs/version_0/metrics.csv"),
                "training_ended_on_step": get_last_step(os.path.join(exp_dir_path, "logs/lightning_logs/version_0/metrics.csv")),
                "scores": read_scores(os.path.join(exp_dir_path, "predictions/all_scores.json")),
                "train_slurm": [],
                "test_slurm": []
            }
            if record["scores"]:
                predictions_dir = os.path.join(exp_dir, "predictions", record["scores"]["checkpoint"].split("/")[-1])
                record["test_predictions"] = os.path.join(predictions_dir, "test_predictions.txt")
                record["val_predictions"] = os.path.join(predictions_dir, "val_predictions.txt")
            
            model_name = exp_dir.split("_TRIAL_s=1000")[0]
            for slurm_f in os.listdir(lang_pair_path):
                slurm_f_path = os.path.join(lang_pair_path, slurm_f)
                if os.path.isfile(slurm_f_path):
                    if slurm_f.endswith(f"TRAIN.{lang_pair}.{model_name}.out"):
                        record["train_slurm"].append(slurm_f_path)
                    elif slurm_f.endswith(f"TEST.{lang_pair}.{model_name}.out"):
                        record["test_slurm"].append(slurm_f_path)
            
            # print(record['train_slurm'])
            # print(record['test_slurm'])
            assert len(record["train_slurm"]) in [0, 1], f"{record['train_slurm']}"
            assert len(record["test_slurm"]) in [0, 1], f"{record['test_slurm']}"

            if len(record["train_slurm"]) > 0:
                record["train_slurm"] = record["train_slurm"][0]
            else:
                record["train_slurm"] = None
            
            if len(record["test_slurm"]) > 0:
                record["test_slurm"] = record["test_slurm"][0]
            else:
                record["test_slurm"] = None

            records[lang_pair].append(record)



100%|██████████| 19/19 [00:12<00:00,  1.57it/s]


In [3]:
from IPython.display import display, HTML

def display_losses(record, width=200):
    html = "<div style='display:flex; gap:20px; flex-wrap:wrap;'>"
    model = record["model"]
    lang_pair = record["lang_pair"]
    html += f"""
    <div style="text-align:left;">
    <div style="font-size:12px; margin-bottom:4px;">{lang_pair} | {model}</div>
    <div style="font-size:12px; margin-bottom:4px;">BEST CHECKPOINT: {record['scores']['checkpoint']}</div>
    <div style="font-size:12px; margin-bottom:4px;">test_BLEU: {record['scores']['test_BLEU']}</div>
    <div style="font-size:12px; margin-bottom:4px;">test_chrF: {record['scores']['test_chrF']}</div>
    <div style="font-size:12px; margin-bottom:4px;">Training Ended on Step: {record['training_ended_on_step']}</div>
    </div>
    """
    for key in ["val_step_loss", "train_step_loss", "epoch_loss"]:
        label = f"{key}"
        html += f"""
        <div style="text-align:center;">
            <div style="font-size:12px; margin-bottom:4px;">{label}</div>
            <img src="{record[key]}" width="{width}">
        </div>        
        """
    html += "</div>"
    display(HTML(html))

def display_losses_without_image(record):
    html = "<div style='display:flex; gap:20px; flex-wrap:wrap;'>"
    model = record["model"]
    lang_pair = record["lang_pair"]
    html += f"""
    <div style="text-align:left;">
    <div style="font-size:12px; margin-bottom:4px;">{lang_pair} | {model}</div>
    <div style="font-size:12px; margin-bottom:4px;">BEST CHECKPOINT: {record['scores']['checkpoint']}</div>
    <div style="font-size:12px; margin-bottom:4px;">test_BLEU: {record['scores']['test_BLEU']}</div>
    <div style="font-size:12px; margin-bottom:4px;">test_chrF: {record['scores']['test_chrF']}</div>
    <div style="font-size:12px; margin-bottom:4px;">Training Ended on Step: {record['training_ended_on_step']}</div>
    </div>
    """
    html += "</div>"
    display(HTML(html))


def spacer(px=20):
    display(HTML(f"<div style='height:{px}px;'></div>"))

def line_spacer(margin=30):
    display(HTML(f"<hr style='margin:{margin}px 0;'>"))

In [4]:
def display_lang_pair_losses(lang_pair):
    for record in sorted(records[lang_pair], key=lambda r: r['model']):
        display_losses(record)
        line_spacer()

def display_file(f, n_lines=(0,0)):
    if n_lines == (0,0):
        return
    head_n_lines, tail_n_lines = n_lines
    print("DISPLAY FILE:", f)
    with open(f) as inf:
        lines = inf.readlines()
    display_top_n(lines, head_n_lines)
    print("\n...\n")
    display_top_n(lines, -tail_n_lines)

def display_top_n(lines, n):
    if n >= 0:
        display_lines = lines[:n]
    else:
        display_lines = lines[n:]
    for line in display_lines:
        print(line, end="")

def display_models(model_prefix=None, langs=["mfe-en", "mfy-eny", "mfx-enx"], REVERSE=False, train=(0,0), test=(0,0)):
    if model_prefix in ["FINETUNE", "PRETRAIN"]:
        model_prefix += "."
    for lang_pair in langs:
        for record in sorted(records[lang_pair], key=lambda r: r['model']):
            if model_prefix and record["model"].startswith(model_prefix) and not record["model"].startswith(model_prefix + "SC_"):
                if REVERSE and "REVERSE" in record["model"]:
                    display_losses_without_image(record)
                    display_file(record["train_slurm"], n_lines=train)
                    line_spacer()
                elif (not REVERSE) and "REVERSE" not in record["model"]:
                    display_losses_without_image(record)
                    display_file(record["train_slurm"], n_lines=train)
                    line_spacer()

                
                



In [5]:
# display_lang_pair_losses("mfx-enx")

In [7]:
display_models(model_prefix="PRETRAIN.SC", langs=["mfe-en", "mfy-eny", "mfx-enx"], REVERSE=False, train=(400,100))

DISPLAY FILE: /home/hatch5o6/nobackup/archive/CognateMT/PredictCognates/archive/CoNLL/mfe-en/9444194_TRAIN.mfe-en.PRETRAIN.SC_fr2mfe-en.out
Fri Jan 23 16:02:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  |   00000000:07:00.0 Off |                    0 |
| N/A   36C    P0             62W /  400W |       0MiB /  81920MiB |      0%      Default |


DISPLAY FILE: /home/hatch5o6/nobackup/archive/CognateMT/PredictCognates/archive/CoNLL/mfy-eny/10322195_TRAIN.mfy-eny.PRETRAIN.SC_fry2mfy-eny.out
Sat Feb 14 17:34:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  |   00000000:84:00.0 Off |                    0 |
| N/A   27C    P0             58W /  400W |       0MiB /  81920MiB |      0%      Defau

DISPLAY FILE: /home/hatch5o6/nobackup/archive/CognateMT/PredictCognates/archive/CoNLL/mfx-enx/10322193_TRAIN.mfx-enx.PRETRAIN.SC_frx2mfx-enx.out
Sat Feb 14 17:32:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  |   00000000:07:00.0 Off |                    0 |
| N/A   27C    P0             62W /  400W |       0MiB /  81920MiB |      0%      Defau